### Importing modules/libraries


In [ ]:
import pandas as pd
import numpy as np
from transformers import pipeline
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pickle

###

###Loading chunks and embeddings files

In [ ]:
with open("/content/combined_chunks.pkl", "rb") as f: #Change path for combined_chunks.pkl
    combined_chunk = pickle.load(f)
embeddings = np.load("/content/embeddings.npy")       #Change path for embeddings.npy

print(f"Total chunks: {len(combined_chunk)}")
print(f"Embeddings shape: {embeddings.shape}")


Total chunks: 15150
Embeddings shape: (15150, 384)


### Emotion Recognition Setup & Data Pre-processing
Loads the emotion classification model (distilroberta-base) and processes all text chunks to identify their emotional tone. It includes a truncation helper to prevent token errors and a utility function to inspect the emotion scores of specific chunks.

In [ ]:
from transformers import pipeline
emotion_model = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    return_all_scores=False,
    device=0  # -1 for CPU
)
def truncate_text(text, max_chars=512):
    return text[:max_chars]

all_chunk = combined_chunk
all_result = []

for chunk in all_chunk:
    truncated_chunk = truncate_text(chunk)
    result = emotion_model(truncated_chunk)[0]
    all_result.append(result)

def check_emotion_score(chunk_index=0):
    res = all_result[chunk_index]
    print(f"Chunk text: {all_chunk[chunk_index][:200]}")
    print(f"Top Emotion: {res['label']} | Score: {res['score']:.3f}")

check_emotion_score(40)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/329M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Chunk text: how so?
Top Emotion: surprise | Score: 0.773


### Emebdder object using Sentence Transformer model

In [ ]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Main Logic
Initializes the text generation model (TinyLlama-1.1B) and defines the core chat_with_user function. This function orchestrates the pipeline: detecting user emotion, retrieving relevant context via cosine similarity, filtering unrelated queries (threshold < 0.45), and generating a human-like response using specific prompt engineering.

In [ ]:
from transformers import pipeline
from sklearn.metrics.pairwise import cosine_similarity
import re

generator = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    torch_dtype="auto",
    device_map="auto",
    max_new_tokens=300,
    temperature=0.3,
    top_p=0.85,
    do_sample=True,
    repetition_penalty=1.2
)

emotion_model = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    return_all_scores=False,
    device_map="auto"
)

SIMILARITY_THRESHOLD = 0.45

def abstract_context(chunks):
    insights = []
    for text in chunks:
        t = text.lower()
        if "anxiety" in t:
            insights.append("Anxiety can cause excessive worry and physical discomfort.")
        if "depression" in t:
            insights.append("Depression may bring sadness, low energy, or emptiness.")
        if "stress" in t:
            insights.append("Stress can make the mind feel overwhelmed and restless.")
        if "panic" in t:
            insights.append("Panic responses can feel sudden and intense.")
        if "fear" in t:
            insights.append("Fear can activate strong emotional reactions.")
    return list(set(insights))[:3]

def chat_with_user(user_input, history):

    user_emotion = emotion_model(user_input)[0]["label"]

    input_embedding = embedder.encode([user_input])
    similarities = cosine_similarity(input_embedding, embeddings)[0]

    top_indices = similarities.argsort()[-4:][::-1]
    top_score = similarities[top_indices[0]]

    if top_score < SIMILARITY_THRESHOLD:
        return "I can't help with that. I am here to support your mental well-being only."

    retrieved_chunks = [combined_chunk[i] for i in top_indices]
    context_points = abstract_context(retrieved_chunks)

    formatted_prompt = f"""<|system|>
You are a empathetic mental health assistant.
CONTEXT: {context_points}
USER EMOTION: {user_emotion}
Instructions: Respond gently. Do NOT repeat the emotion.
</s>
<|user|>
{user_input}
</s>
<|assistant|>"""

    generated_result = generator(
        formatted_prompt,
        max_new_tokens=300,
        pad_token_id=generator.tokenizer.eos_token_id
    )

    full_text = generated_result[0]["generated_text"]

    if "<|assistant|>" in full_text:
        response_only = full_text.split("<|assistant|>")[-1]
    else:
        response_only = full_text[len(formatted_prompt):]

    clean_lines = []
    for line in response_only.split('\n'):
        line = line.strip()
        if not line: continue

        if "USER EMOTION" in line or "CONTEXT:" in line:
            continue
        if "Here is a response" in line or "Sure, I understand" in line:
            continue

        line = re.sub(r'(?i)^assistant:\s*', '', line)

        clean_lines.append(line)

    final_response = " ".join(clean_lines)

    last_punct_index = max(final_response.rfind('.'), final_response.rfind('!'), final_response.rfind('?'))

    if last_punct_index != -1:
        final_response = final_response[:last_punct_index+1]

    return final_response

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'top_p', 'do_sample', 'max_new_tokens', 'repetition_penalty', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Frontend using Gradio
Sets up the frontend using Gradio with custom CSS for a calming theme. It launches the ChatInterface with a public link (share=True), connecting the backend logic to a user-friendly web interface.

In [ ]:
import gradio as gr

custom_css = """
.gradio-container { background-color: #fdf6f0; font-family: 'Inter', sans-serif; }
"""

chat_ui = gr.ChatInterface(
    fn=chat_with_user,
    title="AI Mental Health Support Chatbot",
    description="A supportive AI chatbot for emotional well-being.",
    theme=gr.themes.Soft(),
    css=custom_css
)

chat_ui.launch(share=True)